In [1]:
# Core
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.preprocessing import StandardScaler

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Metrics
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report,
    RocCurveDisplay, PrecisionRecallDisplay, brier_score_loss
)

# Interpretability
import shap

# Settings
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = [10, 6]
plt.rcParams['font.size'] = 12

print("All imports successful")

All imports successful


In [2]:
df = pd.read_csv('heart_disease_processed.csv')
print(df.shape)
print(df.head())
print(df.isnull().sum())

(920, 12)
    age  sex   cp  trestbps   chol  fbs  restecg  thalch  exang  oldpeak  \
0  63.0  1.0  3.0     145.0  233.0  1.0      0.0   150.0    0.0      2.3   
1  67.0  1.0  0.0     160.0  286.0  0.0      0.0   108.0    1.0      1.5   
2  67.0  1.0  0.0     120.0  229.0  0.0      0.0   129.0    1.0      2.6   
3  37.0  1.0  2.0     130.0  250.0  0.0      1.0   187.0    0.0      3.5   
4  41.0  0.0  1.0     130.0  204.0  0.0      0.0   172.0    0.0      1.4   

   slope  target  
0    0.0     0.0  
1    1.0     1.0  
2    1.0     1.0  
3    0.0     0.0  
4    2.0     0.0  
age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalch      0
exang       0
oldpeak     0
slope       0
target      0
dtype: int64


In [3]:
# Separate features and target
X = df.drop('target', axis=1)
y = df['target']

# Hold-out set: 20% reserved for final evaluation
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y  # maintains class distribution
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nClass distribution in training set:")
print(y_train.value_counts(normalize=True).round(3))
print(f"\nClass distribution in test set:")
print(y_test.value_counts(normalize=True).round(3))

Training set: (736, 11)
Test set: (184, 11)

Class distribution in training set:
target
1.0    0.553
0.0    0.447
Name: proportion, dtype: float64

Class distribution in test set:
target
1.0    0.554
0.0    0.446
Name: proportion, dtype: float64


## Scaling Strategy

**StandardScaler** was applied to normalize all features to mean=0 
and standard deviation=1, but only for models that require it.

**Why Logistic Regression needs scaling:**
Logistic Regression assigns a coefficient (weight) to each feature. 
If `age` ranges from 28-77 and `exang` ranges from 0-1, the model 
may assign a disproportionately large coefficient to `exang` simply 
because its values are small — not because it is more important. 
Scaling ensures all features are on the same scale so coefficients 
are comparable and meaningful.

**Why tree-based models do NOT need scaling:**
Random Forest and XGBoost make decisions based on thresholds — 
"if age > 55, go right". The threshold adjusts automatically 
regardless of the feature's scale. Scaling has no effect on 
tree-based model performance.

| Model | Scaling Required |
|-------|-----------------|
| Logistic Regression |  Yes — StandardScaler applied |
| Decision Tree |  No |
| Random Forest |  No |
| XGBoost |  No |

**Implementation note:** The scaler was fit **only on the training set** 
(`fit_transform`) and then applied to the test set (`transform`). 
Fitting on the full dataset would constitute data leakage — the model 
would have indirect knowledge of the test set distribution.

In [4]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Logistic Regression w/ Stratified K-Fold